# Federated Learning for Privacy-Preserving Medical Diagnosis — Colab Training

Real-data run of the **core architecture** on NIH ChestX-ray14 (GPU):
centralised ceiling + single-site floor + **federated FedAvg** global model.
Defence layers (DP, robust aggregation, attacks) are OFF — this is the core
floor / federated / ceiling comparison (Tasks 4, 5, 6).

**Runtime:** ~1–2 h on a Colab T4. Set the runtime to **GPU** first:
`Runtime -> Change runtime type -> T4 GPU`.

**What it does, end to end:**
1. install the federated/DP stack
2. clone this repo
3. download a slice of NIH ChestX-ray14 from Kaggle (CSVs + 2–3 image zips, ~10 GB — not the full 45 GB)
4. run the core suite on a patient-aware ~20k-image subset
5. render charts + a short report, and copy everything to your Google Drive


## 1. Check the GPU

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## 2. Install dependencies

torch / torchvision are preinstalled on Colab — we do **not** reinstall them. We add
Flower **core** (no Ray; the in-process orchestrator is used), Opacus, and a few
utilities. Takes ~1 min.

In [ ]:
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer pandas pillow
print("deps installed")

## 3. Clone the project repo

Make sure the latest code is pushed to GitHub first (including the
`src/experiments/data_setup.py` fix). If the repo is private, replace the URL with a
token form: `https://<TOKEN>@github.com/<user>/<repo>.git`.

In [ ]:
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/content/fl"):
    !git clone -q $REPO_URL /content/fl
%cd /content/fl
!git log --oneline -1

## 4. Kaggle credentials

The dataset comes from Kaggle, which needs an API token:
1. kaggle.com -> your profile -> **Settings** -> **API** -> **Create New Token**.
2. This downloads `kaggle.json`.
3. Run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files
import os, json, pathlib
up = files.upload()                       # choose kaggle.json
pathlib.Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!pip install -q kaggle
print("kaggle configured")

## 5. Download a slice of NIH ChestX-ray14

The full dataset is 45 GB / 112k images across `images_001.zip`…`images_012.zip`.
We only need a slice, so we download the two label CSVs plus a few image zips. Three
zips (`001`+`002`+`003`) give ~25k images; the pipeline then subsamples to ~20k
(patient-aware, keeping the bounding-box / severity images it finds).

Add more zips to `IMAGE_ZIPS` for a bigger run (each extra zip ~10k images, ~4 GB).

In [ ]:
import os, zipfile, glob
DATA = "/content/nih"
os.makedirs(DATA, exist_ok=True)
DS = "nih-chest-xrays/data"
IMAGE_ZIPS = ["images_001.zip", "images_002.zip", "images_003.zip"]

# label + bounding-box CSVs (tiny)
for f in ["Data_Entry_2017.csv", "BBox_List_2017.csv"]:
    if not os.path.exists(f"{DATA}/{f}"):
        !kaggle datasets download -d $DS -f "$f" -p $DATA
        # Kaggle may deliver small files un-zipped or as <name>.zip
        if os.path.exists(f"{DATA}/{f}.zip"):
            with zipfile.ZipFile(f"{DATA}/{f}.zip") as z: z.extractall(DATA)
            os.remove(f"{DATA}/{f}.zip")

# image zips
for z in IMAGE_ZIPS:
    zp = f"{DATA}/{z}"
    if not os.path.exists(zp):
        print("downloading", z, "...")
        !kaggle datasets download -d $DS -f "$z" -p $DATA
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(DATA)
    os.remove(zp)       # free disk after extracting

n_png = len(glob.glob(f"{DATA}/**/*.png", recursive=True))
print("CSVs:", os.path.exists(f"{DATA}/Data_Entry_2017.csv"),
      os.path.exists(f"{DATA}/BBox_List_2017.csv"))
print("images on disk:", n_png)

## 6. Configure the run

Load `configs/colab_real.yaml` and point it at the Colab paths. Tweak
`subsample.max_images`, `training.local_epochs`, or `training.rounds` to trade
runtime for quality.

In [ ]:
import sys; sys.path.insert(0, "/content/fl")
from src.experiments.config import load_config

cfg = load_config("configs/colab_real.yaml")
cfg["data"]["root"] = DATA
cfg["partition"]["manifest_path"] = f"{DATA}/manifest.json"
cfg["data"]["subsample"]["max_images"] = 20000   # raise for a bigger run
cfg["training"]["local_epochs"] = 3              # baseline epochs
cfg["training"]["rounds"] = 10                   # federated rounds
print("image_size:", cfg["data"]["image_size"], "| subsample:", cfg["data"]["subsample"])

## 7. Run the core suite

Centralised -> single-site -> federated, all on the same partition and test set.
Progress prints per epoch / per round. This is the long cell (~1–2 h).

In [ ]:
from src.experiments.suite import run_core_suite

OUT = "/content/fl/results/colab"
summary = run_core_suite(
    cfg, out_dir=OUT, data_root=DATA, fed_local_epochs=1,
    extra_setup={"dataset": "NIH ChestX-ray14 (subset)",
                 "image_size": cfg["data"]["image_size"],
                 "baseline_epochs": cfg["training"]["local_epochs"]},
)

## 8. Charts + report

In [ ]:
from src.evaluation.report import generate_report
generate_report(OUT)

from IPython.display import Image, display, Markdown
for p in ["03_floor_fed_ceiling", "01_learning_curve", "02_global_loss", "04_per_pathology_auc"]:
    display(Image(f"{OUT}/plots/{p}.png"))
display(Markdown(open(f"{OUT}/REPORT.md").read()))

## 9. Save results to Google Drive

Copies results (JSON + plots + report) to your Drive so they survive the runtime
shutting down. Download them and commit into `results/` in the repo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, os
dst = "/content/drive/MyDrive/fl-medical-results/colab"
os.makedirs(os.path.dirname(dst), exist_ok=True)
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(OUT, dst)
print("copied results ->", dst)